# Get the Labels

In [1]:
import numpy as np
from astropy.io import fits # You might need to pip install this
import pylab as plt # only needed for verification

In [2]:
path_labels = "./labels.fits"
allstar = fits.open(path_labels)

FileNotFoundError: [Errno 2] No such file or directory: './labels.fits'

In [ ]:
# the labels are in an enormous table in element [1] of this FITS file
labels = allstar[1].data
plt.scatter(labels['TEFF'], labels['LOGG'], s=1)
plt.xlim(6000, 3500)
plt.ylim(5, 0)

In [ ]:
# make a reasonable red-giant-branch sample
RGB = True
RGB = np.logical_and(RGB, labels['TEFF'] > 3500.)
RGB = np.logical_and(RGB, labels['TEFF'] < 5400.)
RGB = np.logical_and(RGB, labels['LOGG'] < 3.0)
RGB = np.logical_and(RGB, labels['LOGG'] > 0.0)
RGB = np.logical_and(RGB, labels['H'] < 10.5)
print(np.sum(RGB))

In [ ]:
# make a plot that an astronomer likes to see
RGB_labels = labels[RGB]
plt.scatter(RGB_labels['TEFF'], RGB_labels['LOGG'], c=RGB_labels['FE_H'], s=1)
plt.xlim(5400, 3500)
plt.xlabel("effective temperature")
plt.ylim(3., 0.)
plt.ylabel("log10 surface gravity")
plt.colorbar(label="metallicity")

In [ ]:
# make train, validation, and test data sets
rng = np.random.default_rng(17)
N_RGB = len(RGB_labels)
N_train, N_valid, N_test = 1024, 256, 512
I = rng.permutation(N_RGB)
I_train = I[0:N_train]
I_valid = I[N_train:N_train+N_valid]
I_test = I[N_train+N_valid:N_train+N_valid+N_test]

train_labels = RGB_labels[I_train]
valid_labels = RGB_labels[I_valid]
test_labels = RGB_labels[I_test]
print(len(train_labels), len(valid_labels), len(test_labels))

Here how you get `LOGG`

In [ ]:
train_labels_logg = train_labels['LOGG']
print(train_labels_logg.shape) # (num_spectra, 1)

# Get the Features

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
train_features = np.load('./train_features.npy')
valid_features = np.load('./valid_features.npy')
test_features = np.load('./test_features.npy')
for i in range(10):
    plt.plot(train_features[i] + i)

In [ ]:
print(train_features.shape) # (num_spectra, num_pixels)

# Split ; Predict `LOGG`

In [ ]:
N_train, N_valid, N_test = 1024, 256, 512

#RGB_labels
rng = np.random.default_rng(17)
N_RGB = len(RGB_labels)
I = rng.permutation(N_RGB)

In [ ]:
y_train = train_labels["LOGG"].astype(np.float32)
y_valid = valid_labels["LOGG"].astype(np.float32)
y_test  = test_labels["LOGG"].astype(np.float32)

print("y shapes:", y_train.shape, y_valid.shape, y_test.shape)
print("LOGG ranges:",
      (float(np.min(y_train)), float(np.max(y_train))),
      (float(np.min(y_valid)), float(np.max(y_valid))),
      (float(np.min(y_test)),  float(np.max(y_test))))

In [ ]:
train_features = np.load("./train_features.npy").astype(np.float32)
valid_features = np.load("./valid_features.npy").astype(np.float32)
test_features  = np.load("./test_features.npy").astype(np.float32)

X_train = np.nan_to_num(train_features)
X_valid = np.nan_to_num(valid_features)
X_test  = np.nan_to_num(test_features)

print("X shapes:", X_train.shape, X_valid.shape, X_test.shape)

assert X_train.shape[0] == y_train.shape[0], "Train X/y mismatch"
assert X_valid.shape[0] == y_valid.shape[0], "Valid X/y mismatch"
assert X_test.shape[0]  == y_test.shape[0],  "Test X/y mismatch"

#(OpenAI, 2026)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, r2

def report(name, split, y_true, y_pred):
    mse, rmse, r2 = metrics(y_true, y_pred)
    print(f"{name:>18} | {split:>5} | MSE={mse:.4f}  RMSE={rmse:.4f}  R^2={r2:.4f}")

def pred_vs_true_plot(name, y_true, y_pred):
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, s=8, alpha=0.5)
    lo = min(float(y_true.min()), float(y_pred.min()))
    hi = max(float(y_true.max()), float(y_pred.max()))
    plt.plot([lo, hi], [lo, hi], 'k--', lw=2)
    plt.xlabel("True LOGG")
    plt.ylabel("Predicted LOGG")
    plt.title(f"{name}: Predicted vs True (TEST)")
    plt.grid(True, alpha=0.3)
    plt.show()

#(OpenAI, 2026)

In [ ]:
# (a) Linear Regression:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

lr_valid = lr.predict(X_valid_scaled)
lr_test  = lr.predict(X_test_scaled)

report("Linear Regression", y_valid, lr_valid, "VALID")
report("Linear Regression", y_test,  lr_test,  "TEST")
plot_pred_vs_true("Linear Regression", y_test, lr_test, "TEST")

In [ ]:
# (b) KNN:
k_grid = [3, 5, 10, 20, 40]
best_k, best_rmse = None, np.inf

for k in k_grid:
    knn = KNeighborsRegressor(n_neighbors=k, weights="distance")
    knn.fit(X_train_scaled, y_train)
    pred = knn.predict(X_valid_scaled)
    _, rmse, _ = metrics(y_valid, pred)
    print(f"k={k:>2}  valid RMSE={rmse:.4f}")
    if rmse < best_rmse:
        best_rmse = rmse
        best_k = k

knn_model = KNeighborsRegressor(n_neighbors=best_k, weights="distance")
knn_model.fit(X_train_scaled, y_train)

knn_valid = knn_model.predict(X_valid_scaled)
knn_test  = knn_model.predict(X_test_scaled)

print(f"Chosen best k = {best_k}")
report(f"KNN (k={best_k})", "valid", y_valid, knn_valid)
report(f"KNN (k={best_k})", "test",  y_test,  knn_test)
pred_vs_true_plot(f"KNN (k={best_k})", y_test, knn_test)

#(OpenAI, 2026)

In [ ]:
# (c) MLP
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)

mlp_valid = mlp.predict(X_valid_scaled)
mlp_test  = mlp.predict(X_test_scaled)

report("MLP (128,64)", y_valid, mlp_valid, "VALID")
report("MLP (128,64)", y_test,  mlp_test,  "TEST")
plot_pred_vs_true("MLP (128,64)", y_test, mlp_test, "TEST")

# Hyperparameter

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

linear_candidates = [
    ("LinearRegression", LinearRegression()),
    ("Ridge(alpha=0.1)", Ridge(alpha=0.1, random_state=42)),
    ("Ridge(alpha=1.0)", Ridge(alpha=1.0, random_state=42)),
    ("Ridge(alpha=10)",  Ridge(alpha=10.0, random_state=42)),
]

best_linear = None
best_rmse = np.inf

for name, model in linear_candidates:
    model.fit(X_train_scaled, y_train)
    pred_valid = model.predict(X_valid_scaled)
    m = compute_metrics(y_valid, pred_valid)
    print_metrics(name, "valid", m)
    if m["rmse"] < best_rmse:
        best_rmse = m["rmse"]
        best_linear = (name, model)

print("\nBest linear model:", best_linear[0])

In [ ]:
lr_test = lr.predict(X_test_scaled)

mse = mean_squared_error(y_test, lr_test)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, lr_test)

print(f"Linear Regression TEST RMSE: {rmse:.4f}, R^2: {r2:.4f}")

In [ ]:
# (b) KNN:

best_knn_params = {"k": 5, "weights": "distance"}

def knn_bootstrap_seed_test(k, weights, seeds=(17, 42, 99), frac=0.9):
    rmses = []
    n = X_train_scaled.shape[0]
    m = int(frac * n)

    for s in seeds:
        rng = np.random.default_rng(s)
        idx = rng.choice(n, size=m, replace=True)

        knn = KNeighborsRegressor(n_neighbors=k, weights=weights)
        knn.fit(X_train_scaled[idx], y_train[idx])
        pred = knn.predict(X_valid_scaled)

        rmse = np.sqrt(mean_squared_error(y_valid, pred))
        rmses.append(rmse)

        print(f"Bootstrap seed={s:>2} | Valid RMSE={rmse:.4f}")

    print(f"\nKNN bootstrap RMSE mean ± std = "
          f"{float(np.mean(rmses)):.4f} ± {float(np.std(rmses)):.4f}")

In [ ]:
#Rerun the algorithm but with various RNG seeds
knn_bootstrap_seed_test(**best_knn_params)


best_knn_model = KNeighborsRegressor(
    n_neighbors=best_knn_params["k"],
    weights=best_knn_params["weights"]
)
best_knn_model.fit(X_train_scaled, y_train)

# --- Evaluate ---
knn_valid = best_knn_model.predict(X_valid_scaled)
knn_test  = best_knn_model.predict(X_test_scaled)

report("Best KNN", "valid", y_valid, knn_valid)
report("Best KNN", "test",  y_test,  knn_test)

# Plot
pred_vs_true_plot(f"Best KNN (k={best_knn_params['k']})", y_test, knn_test)

#(OpenAI, 2026)

In [ ]:
# (c) MLP
seeds_to_test = [17, 42, 99]
seed_results = []

for seed in seeds_to_test:
    mlp_model = MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=300,
        early_stopping=True,
        random_state=seed
    )

    mlp_model.fit(X_train_scaled, y_train)
    pred = mlp_model.predict(X_valid_scaled)
    _, rmse, _ = metrics(y_valid, pred)

    seed_results.append((seed, rmse, mlp_model))
    print(f"MLP Seed={seed:>2} | Validation RMSE={rmse:.4f}")

rmses = np.array([r for (_, r, _) in seed_results], dtype=float)
print(f"\nValidation RMSE across seeds: mean ± std = {rmses.mean():.4f} ± {rmses.std():.4f}")

# Choose a fixed seed for reproducibility
chosen_seed = 42
best_mlp_model = [m for (s, r, m) in seed_results if s == chosen_seed][0]

print(f"=> Using seed = {chosen_seed} ")

#(OpenAI, 2026)

In [ ]:
mlp_valid = best_mlp_model.predict(X_valid_scaled)
mlp_test  = best_mlp_model.predict(X_test_scaled)

report(f"MLP (seed={chosen_seed})", "valid", y_valid, mlp_valid)
report(f"MLP (seed={chosen_seed})", "test",  y_test,  mlp_test)

pred_vs_true_plot(f"MLP (seed={chosen_seed})", y_test, mlp_test)

The MLP shows minor variation across random seeds due to stochastic optimization, but the validation RMSE remains within a narrow range. This suggests the training process converges to similar solutions rather than drastically different outcomes.